# III. Comparaison des triptyques

Comparaison par identifiants globaux de tokens "normalisés" (`AlignID_*`).


In [8]:
import pandas as pd

TRIPTYQUES_FOLDER = "../results/csv_triptyques/"

auto = pd.read_csv(f"{TRIPTYQUES_FOLDER}auto_chap1to5_chap.csv")
manuel = pd.read_csv(f"{TRIPTYQUES_FOLDER}manuel_chap1to5_chap.csv")


## 1) Comparaison des triptyques seuls
Seulement avec les id de tokens de Sujet/Verbe/objet

In [9]:
auto_chap1to5 = auto[auto["chapter"].isin([1, 2, 3, 4, 5])].copy()

print("Triptyques automatiques chap. 1 à 5 :", len(auto_chap1to5))
print("Triptyques manuels :", len(manuel))

key_cols = ["AlignID_sujet", "AlignID_verbe", "AlignID_objet"]

missing_auto = [c for c in key_cols if c not in auto_chap1to5.columns]
missing_manuel = [c for c in key_cols if c not in manuel.columns]

if missing_auto or missing_manuel:
    raise ValueError(f"Colonnes manquantes - auto: {missing_auto}, manuel: {missing_manuel}")

Triptyques automatiques chap. 1 à 5 : 1255
Triptyques manuels : 1255


#### Création d'un clé unique par triptyque

avec ID de sujet/verbe/objet

In [10]:
for df in [auto_chap1to5, manuel]:
    df["cle_triptyque"] = (df[key_cols].fillna("").astype(str).apply(lambda row: " || ".join(row), axis=1))

#### Comparaison de ces triptyques avec cette clé

In [11]:
cles_auto = set(auto_chap1to5["cle_triptyque"])
cles_manuel = set(manuel["cle_triptyque"])

communs = cles_auto & cles_manuel
seulement_auto = cles_auto - cles_manuel
seulement_manuel = cles_manuel - cles_auto

print("Triptyques communs :", len(communs))
print("Triptyques seulement automatiques :", len(seulement_auto))
print("Triptyques seulement manuels :", len(seulement_manuel))


Triptyques communs : 1253
Triptyques seulement automatiques : 0
Triptyques seulement manuels : 0


In [12]:
df_communs = auto_chap1to5[auto_chap1to5["cle_triptyque"].isin(communs)].copy()

df_seulement_auto = auto_chap1to5[auto_chap1to5["cle_triptyque"].isin(seulement_auto)].copy()
df_seulement_auto["source_comparaison"] = "seulement_automatique"

df_seulement_manuel = manuel[manuel["cle_triptyque"].isin(seulement_manuel)].copy()
df_seulement_manuel["source_comparaison"] = "seulement_manuel"

differences_triptyques = pd.concat(
    [df_seulement_auto, df_seulement_manuel],
    ignore_index=True
)

print("Communs :", len(df_communs))
print("Différences :", len(differences_triptyques))

differences_triptyques[
    ["source_comparaison", "Phrase", "Sujet", "Verbe", "Objet",
     "AlignID_sujet", "AlignID_verbe", "AlignID_objet",
     "Dep_sujet", "Dep_verbe", "Dep_objet"]
].head(100)

Communs : 1255
Différences : 0


,source_comparaison,Phrase,Sujet,Verbe,Objet,AlignID_sujet,AlignID_verbe,AlignID_objet,Dep_sujet,Dep_verbe,Dep_objet


## 2) Comparaison avec les entités

In [13]:
for df in [auto_chap1to5, manuel]:
    df["a_PER_FAC_triptyque"] = (
        (df["a_personnage_triptyque"] == True)
        | df["types_lieux_triptyque"].fillna("").astype(str).str.contains("FAC"))
    
    df["a_LOC_triptyque"] = (df["types_lieux_triptyque"].fillna("").astype(str).str.contains("LOC"))

key_cols_entites = ["AlignID_sujet","AlignID_verbe","AlignID_objet","a_PER_FAC_triptyque", "a_LOC_triptyque"]

for df in [auto_chap1to5, manuel]:
    df["cle_triptyque_entites"] = (df[key_cols_entites].fillna("").astype(str).apply(lambda row: " || ".join(row), axis=1))

cles_auto_entites = set(auto_chap1to5["cle_triptyque_entites"])
cles_manuel_entites = set(manuel["cle_triptyque_entites"])

communs_entites = cles_auto_entites & cles_manuel_entites
seulement_auto_entites = cles_auto_entites - cles_manuel_entites
seulement_manuel_entites = cles_manuel_entites - cles_auto_entites

print("Triptyques + PER/FAC/LOC communs :", len(communs_entites))
print("Triptyques + PER/FAC/LOC seulement automatiques :", len(seulement_auto_entites))
print("Triptyques + PER/FAC/LOC seulement manuels :", len(seulement_manuel_entites))

Triptyques + PER/FAC/LOC communs : 1148
Triptyques + PER/FAC/LOC seulement automatiques : 105
Triptyques + PER/FAC/LOC seulement manuels : 105
